# 현금영수증 한 건(들)의 전체 반환 필드 확인

같은 날짜 범위·단말기·GBN=2(현금영수증) 조건으로 승인번호별 한 페이지씩 조회한 뒤,
반환된 모든 필드를 가로 표로, 선택한 거래는 세로 비교표와 원본 JSON으로 확인한다.

1. 아래 0번 설정에서 `SDATE`/`EDATE`를 실제 발급일로, `AUTHNOS`를 승인번호 목록(예: `["승인번호1", "승인번호2"]`)으로 바꾼다.
   번호는 따옴표 안에 넣어 앞자리 0을 유지한다. 승인번호를 모르면 `AUTHNOS = []`로 두고 해당 날짜의 목록에서 고른다.
2. 위에서부터 실행하고 4번 결과를 확인한다.
3. 4-1번의 `ROW_INDEX`에 행 번호를, 4-2번의 `SELECTED_AUTHNOS`에 비교할 승인번호를 넣고 실행한다.

- 중복 승인번호는 한 번만 조회하며, 번호마다 지정한 한 페이지만 조회한다(전체 페이지 합계가 아님).
- 단말기번호는 `.env`의 `SMARTRO_VAN_TERMID`, 인증키는 `SMARTRO_VAN_API_KEY`에서 읽는다.
- API가 제공하지 않는 정보나 마스킹된 원문을 복원하지 않는다. GET 조회만 하며 발급·취소 요청은 하지 않는다.

공개 명세: https://exttran.smilebiz.co.kr/getApiSvcInfoData?SVC_CTGR=VAN


## 0. 공통 설정 (Bearer 인증 + 요청 함수)

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import display

for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = [parent, parent / "smilebiz-van/van-api"]
    helper_dir = next((p for p in candidates if (p / "van_test_helpers.py").exists()), None)
    if helper_dir is not None:
        sys.path.insert(0, str(helper_dir))
        break
else:
    raise RuntimeError("노트북 폴더 또는 저장소 루트에서 실행하세요.")
from van_test_helpers import van_get, rows_of, env_value

# -- 조회 조건 --------------------------------------------------
TERMID = env_value("SMARTRO_VAN_TERMID")   # 단말기번호 10자리 (.env)
SDATE = "20260909"        # 실제 발급일(YYYYMMDD)
EDATE = "20260909"
# 승인번호 목록. 여러 개: ["승인번호1", "승인번호2"]. 날짜별 전체 목록: []
AUTHNOS = []
GBN = "2"                 # 현금영수증
CURRPAGE = 0
STIME = ""
ETIME = ""
COMP_NO = env_value("SMARTRO_VAN_COMP_NO")
COMP_IDX = env_value("SMARTRO_VAN_COMP_IDX")
# ------------------------------------------------------------


def normalize_authnos(values):
    if not isinstance(values, (list, tuple)):
        raise ValueError('AUTHNOS는 ["승인번호1", "승인번호2"] 형식으로 입력하세요.')
    normalized = []
    for value in values:
        if not isinstance(value, str) or not value.strip():
            raise ValueError('승인번호는 비어 있지 않은 문자열이어야 합니다. 전체 목록은 AUTHNOS = []로 설정하세요.')
        value = value.strip()
        if value not in normalized:
            normalized.append(value)
    return normalized


def validate_filters():
    normalize_authnos(AUTHNOS)
    if len(TERMID) != 10:
        raise ValueError("TERMID는 10자리 문자열이어야 합니다. .env의 SMARTRO_VAN_TERMID를 확인하세요.")
    for value in (SDATE, EDATE):
        if len(value) != 8 or not value.isdigit():
            raise ValueError("날짜는 YYYYMMDD 8자리여야 합니다.")
        datetime.strptime(value, "%Y%m%d")
    if SDATE > EDATE:
        raise ValueError("시작일이 종료일보다 늦습니다.")
    for value in (STIME, ETIME):
        if value:
            if len(value) != 6 or not value.isdigit():
                raise ValueError("시간은 hhmmss 6자리 또는 빈 값이어야 합니다.")
            datetime.strptime(value, "%H%M%S")
    if SDATE == EDATE and STIME and ETIME and STIME > ETIME:
        raise ValueError("시작시간이 종료시간보다 늦습니다.")
    if GBN != "2":
        raise ValueError("이 노트북은 현금영수증 GBN=2 전용입니다.")
    if type(CURRPAGE) is not int or CURRPAGE < 0:
        raise ValueError("페이지는 0 이상의 정수여야 합니다.")


def response_rows(data):
    return rows_of(data)


validate_filters()
print(f"단말기: {TERMID} | 현금영수증 | 기간: {SDATE} ~ {EDATE}")
print("인증키 설정됨 (값은 표시하지 않음)")

## 1. 서버연결 상태체크
성공해도 해당 가맹점의 매출 조회 권한까지 확인된 것은 아닙니다.

In [ ]:
check = van_get("/V1/common/serverChecks")
check

## 2. 공통코드정보조회

`GET /V1/common/getCommCodeInfo` — 파라미터 없음. `GBN`(승인구분), `HID_GBN`(카드사구분),
`ORGCOD`(카드사코드) 등 다른 API 호출 시 필요한 코드값을 확인할 수 있습니다.

In [ ]:
comm_codes = van_get("/V1/common/getCommCodeInfo")
cash_codes = [c for g in comm_codes.get("CODE_INFO", []) if g.get("GROUP_CODE") == "GBN"
              for c in g.get("CODES", []) if str(c.get("CODE")) == GBN]
print("현금영수증 코드 확인:", cash_codes)
if not cash_codes:
    raise RuntimeError("공통코드에서 GBN=2를 찾지 못했습니다. 스마트로에 확인하세요.")

## 3. 해당 단말기의 현금영수증 집계
집계 API 응답 중 GBN=2 행을 가로 표로 표시합니다. 집계 행 수와 실제 거래 건수(SALES_CNT)는 다릅니다.


In [ ]:
validate_filters()
sales_sum = van_get("/V1/sales/getSalesSum", params={
    "SDATE": SDATE, "EDATE": EDATE, "COMP_NO": COMP_NO, "COMP_IDX": COMP_IDX, "TERMID": TERMID,
})
sales_sum_df = pd.DataFrame([r for r in response_rows(sales_sum) if str(r.get("GBN")) == GBN])
print(f"현금영수증 집계 그룹: {len(sales_sum_df)}행 (거래 건수가 아닙니다)")
print("실제 승인 건수는 SALES_CNT, 승인 합계는 SALES_AMT입니다. 소득공제·지출증빙은 합산됩니다.")
display(sales_sum_df)

## 4. 여러 승인번호의 현금영수증 승인·취소 내역 (번호별 한 페이지)
0번의 TERMID 설정과 GBN=2로 조회합니다. 날짜·승인번호·페이지는 0번 설정에서 바꾼 후 다시 실행하세요.

기존 노트북에는 서버가 빈 선택 필드도 요구한다는 테스트 기록이 있으므로 모든 요청 필드를 유지합니다.
페이지는 공개 명세에 따라 0부터 시작합니다. 반환되는 CURRPAGE/TOTPAGE를 확인하고 다음 페이지는 직접 지정하세요.
이 표는 전체 기간의 모든 페이지를 합친 결과가 아닙니다. 기존 데이터 없음 응답(VAN_SALE-1001)은 빈 목록으로 안내하고, 그 외 API 오류는 중단합니다.

API 응답의 모든 열을 가로 표로 표시합니다.


In [ ]:
# API는 승인번호 한 개씩 요청합니다. 모든 요청을 검증한 뒤 결과를 합칩니다.
# 실패하면 이전/부분 결과가 상세 표에 사용되지 않도록 초기화합니다.
sales_list = None
sales_list_df = pd.DataFrame()
selected_receipt = None
receipt_detail_df = pd.DataFrame()
validate_filters()

def query_receipts():
    requested_authnos = normalize_authnos(AUTHNOS) or [""]
    combined_rows, responses, row_sources, summaries = [], [], [], []
    for authno in requested_authnos:
        params = {
            "SDATE": SDATE, "STIME": STIME, "EDATE": EDATE, "ETIME": ETIME,
            "COMP_NO": COMP_NO, "COMP_IDX": COMP_IDX, "TERMID": TERMID,
            "GBN": GBN, "CDNO": "", "AUTHNO": authno, "HID_GBN": "", "ORGCOD": "",
            "REJEC_TYPE": "1", "CURRPAGE": CURRPAGE,
        }
        print(f"승인번호 {authno or '미지정(날짜별 목록)'} 조회 중")
        response = van_get("/V1/sales/getSalesList", params=params)
        received = response_rows(response)
        for row in received:
            if row.get("TERMID") not in (None, "") and str(row["TERMID"]).strip() != TERMID:
                raise RuntimeError("요청과 다른 단말기 거래가 반환되었습니다.")
            if row.get("GBN") not in (None, "") and str(row["GBN"]) != GBN:
                raise RuntimeError("현금영수증 외 거래가 반환되었습니다.")
            if authno and str(row.get("AUTHNO", "")).strip() != authno:
                raise RuntimeError("요청과 다른 승인번호가 반환되었습니다.")
        summary = {"요청 승인번호": authno or "미지정", "응답코드": response.get("CODE"),
                   "수신 건수": len(received)}
        for field in ("CURRPAGE", "TOTPAGE", "TOT_CNT", "TOT_AMT"):
            value = response.get(field)
            if value is None and received:
                value = received[0].get(field)
            summary[field] = value
        summaries.append(summary)
        row_sources.extend([len(responses)] * len(received))
        responses.append({"params": params, "response": response})
        combined_rows.extend(received)
    # 로컬에서 합친 구조입니다. 원본 응답은 RESPONSES 안에 그대로 보관합니다.
    return {"DATA": combined_rows, "RESPONSES": responses, "ROW_SOURCES": row_sources}, summaries

sales_list, request_summaries = query_receipts()
sales_list_df = pd.DataFrame(response_rows(sales_list))
print(f"단말기 {TERMID} | {SDATE} ~ {EDATE} | 합계 수신 {len(sales_list_df)}건")
display(pd.DataFrame(request_summaries))
print("각 승인번호의 지정 페이지 한 개씩 조회한 결과입니다. 전체 페이지 합계가 아닙니다.")
if sales_list_df.empty:
    print("조회 결과가 없습니다. 날짜 범위와 승인번호를 확인하세요. 날짜별 목록은 AUTHNOS = []로 설정하세요.")
else:
    with pd.option_context("display.max_rows", None, "display.max_columns", None,
                           "display.max_colwidth", None):
        display(sales_list_df)


## 4-1. 선택한 거래 한 건 — 가로 표와 원본 JSON
ROW_INDEX에 4번 목록의 행 번호를 입력하세요.


In [ ]:
# 4번 목록의 행 번호를 선택합니다. API가 반환한 필드만 가로 표로 표시합니다.
ROW_INDEX = 0
selected_receipt = None
receipt_detail_df = pd.DataFrame()

def show_receipt_detail(sales_list, sales_list_df, row_index):
    if sales_list is None:
        raise RuntimeError("먼저 0번 설정과 4번 조회를 실행하세요.")
    rows = response_rows(sales_list)
    if sales_list_df.empty:
        if rows:
            raise RuntimeError("4번 조회 결과 검증이 완료되지 않았습니다.")
        print("조회 결과가 0건이므로 표시할 상세 정보가 없습니다.")
        return None, pd.DataFrame()
    if type(row_index) is not int or not 0 <= row_index < len(rows):
        raise ValueError(f"ROW_INDEX는 0 ~ {len(rows) - 1} 사이 정수여야 합니다.")
    receipt = rows[row_index]
    source = sales_list["RESPONSES"][sales_list["ROW_SOURCES"][row_index]]["response"]
    detail = pd.DataFrame([receipt])
    with pd.option_context("display.max_rows", None, "display.max_columns", None,
                           "display.max_colwidth", None):
        display(detail)
    print("선택한 거래 원본 JSON")
    print(json.dumps(receipt, ensure_ascii=False, indent=2))
    print("응답 공통 정보 (DATA 제외)")
    print(json.dumps({k: v for k, v in source.items() if k != "DATA"}, ensure_ascii=False, indent=2))
    return receipt, detail

selected_receipt, receipt_detail_df = show_receipt_detail(
    globals().get("sales_list"), globals().get("sales_list_df", pd.DataFrame()), ROW_INDEX
)


## 4-2. 선택한 승인번호 — 마지막 세로 비교표
SELECTED_AUTHNOS에 승인번호를 입력하세요. 행은 API 응답 항목, 열은 거래입니다. 추가 API 호출이나 발급용도 추정은 하지 않습니다.


In [ ]:
# 4번 목록에서 표로 보고 싶은 승인번호를 입력하세요. 여러 개 입력할 수 있습니다.
SELECTED_AUTHNOS = []  # 예: ["승인번호1", "승인번호2"]

selected_receipts_df = pd.DataFrame()

def filter_selected_receipts(frame, authnos):
    selected = normalize_authnos(authnos)
    if not selected:
        print("SELECTED_AUTHNOS에 표로 볼 승인번호를 입력하세요.")
        return frame.iloc[0:0].copy()
    if frame.empty:
        print("4번 조회 결과가 0건입니다. 날짜·단말기·페이지를 확인하세요.")
        return frame.copy()
    if "AUTHNO" not in frame.columns:
        raise RuntimeError("4번 조회 결과에 AUTHNO 필드가 없습니다.")
    # 앞자리 0을 유지하며 정확히 일치하는 번호만 선택합니다.
    approval_numbers = frame["AUTHNO"].astype("string").str.strip()
    result = frame.loc[approval_numbers.isin(selected)].copy()
    found = set(approval_numbers.loc[result.index].dropna())
    missing = [number for number in selected if number not in found]
    if missing:
        print("현재 4번 조회 결과에서 찾지 못한 승인번호:", ", ".join(missing))
        print("해당 번호의 날짜·단말기·페이지를 확인하세요. 이 셀은 추가 API 조회를 하지 않습니다.")
    return result

if globals().get("sales_list") is None or "sales_list_df" not in globals():
    raise RuntimeError("먼저 0번 설정과 4번 조회를 실행하세요. 4-1번은 실행하지 않아도 됩니다.")
selected_receipts_df = filter_selected_receipts(sales_list_df, SELECTED_AUTHNOS)
if not selected_receipts_df.empty:
    print(f"선택한 승인번호의 거래 {len(selected_receipts_df)}건 (승인·취소 등 일치하는 모든 행)")
    # 원본의 모든 열과 4번 행 번호를 유지합니다.
    with pd.option_context("display.max_rows", None, "display.max_columns", None,
                           "display.max_colwidth", None):

        print("세로 비교: 행은 API 응답 항목, 열은 선택한 거래입니다.")
        comparison_df = selected_receipts_df.copy()
        comparison_df.index = [f"{row.get('AUTHNO', '')}"
                               for index, row in comparison_df.iterrows()]
        # 설명은 공개 명세의 필드명 번역입니다. 거래 값을 분류하거나 변경하지 않습니다.
        field_labels = {
            "CURRPAGE": "현재 페이지", "TOTPAGE": "전체 페이지 수",
            "TSDATE": "승인일자", "TSTIME": "승인시간",
            "COMP_NO": "사업자번호", "COMP_IDX": "사업자 IDX", "MEMBNAM": "가맹점명",
            "NRSPC_NM": "승인·취소·거절 구분", "CDNO": "식별번호",
            "ISTMMON": "할부기간", "AMT1": "금액", "AMT2": "봉사료", "AMT3": "부가가치세",
            "AUTHNO": "승인번호", "TERMID": "단말기번호",
            "HID": "발급사코드", "HID_NM": "발급사명",
            "ACQHID": "매입사코드", "ACQHID_NM": "매입사명",
            "DEPDATE": "입금예정일자", "TSFEE": "수수료", "DPAMT": "입금예정액",
            "CHECK_FLAG": "체크카드여부", "ORGDATE": "원거래일자", "MTRCNO": "일련번호",
            "NRSPC_MSG": "승인결과코드·메시지", "HCODE_MSG": "반송응답코드·메시지",
            "ACQBANK": "간편결제명", "TR_TYPE": "카드거래타입 (MS/IC)",
            "GBN": "결제수단 구분코드", "TOT_CNT": "조회 전체 건수", "TOT_AMT": "조회 전체 금액",
        }
        vertical_df = comparison_df.T
        vertical_df.index = pd.MultiIndex.from_tuples(
            [(key, field_labels.get(key, "추가 응답 항목 (설명 미확인)"))
             for key in vertical_df.index], names=["API 코드", "한글 항목명"])
        display(vertical_df)


## 5. (참고) 입금 관련 API

가이드 사이트(VAN 탭)에 아래 API가 더 있지만, 이 노트북에서는 요청 파라미터를 확인하지 않았습니다.
실제로 쓰시려면 https://exttran.smilebiz.co.kr 에서 `VAN` 탭 -> 해당 항목을 눌러 `Request` 표를 직접 확인한 뒤,
위 `van_get()` 함수에 경로와 파라미터만 맞춰서 그대로 재사용하시면 됩니다.

- 입금내역 조회(집계)
- 입금내역 조회(상세)
- 입금보류내역 조회
- 청구내역 조회